# Normalization H1–H4

Run all cells.

Outputs (empatica* and eyetracker* columns only):
- Normalization_H1toH4_all_parameters.csv
- Normalization_H1toH4_diagnostics.csv


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import re
import os
import shutil
import tempfile

# 1. Config
ROOT = Path(r"C:/Users/pandya/Documents/Github/docker/Paper3_Final")
DATA = Path(r"C:/Users/pandya/Documents/Github/docker/Paper3_Final/output/merged_all_11participants_8min.csv")  # same input as H1_to_H3.R / H4_multivariate.R
CORE_PHASES = ["BikeU", "BikeG", "WalkU", "WalkG"]  # x <= 0 counts use the analysed core phases only
PAPERS = Path(r"C:/Users/pandya/Documents/Github/docker/Paper3_Github/Experiment papers")
OUT_DIR = ROOT / "output/statistical_analysis/Normalization & Parameter Screening/Normalization"

# never ln(1 + x). Must stay identical to H4_SPEC$outcome_transform in scripts/H4_multivariate.R.
STEP5_TEXT = {
    "No transformation": "No transformation",
    "Natural log": "Natural log",
}
# Physiological outcomes fitted in H4 Step 5 (H4_SPEC$outcomes in scripts/H4_multivariate.R).
# empatica__eda_scl_usiemens is not a Step 5 outcome.
NOT_IN_STEP5 = {"empatica__eda_scl_usiemens"}
NORMALIZATION_PREFIXES = ("empatica", "eyetracker")

def in_normalization_scope(column: str) -> bool:
    return column.startswith(NORMALIZATION_PREFIXES)

PAPER_GROUP_1 = ["Fowles", "EDA_recommendations_BIOPAC", "Vocks et al 2021", "Zhang et al 2020", "Figner_Murphy_2010"]
PAPER_GROUP_2 = ["Laborde Mosley Thayer 2017", "An Overview of Heart Rate Variability Metrics and Norms", "HRV_task_force", "Reliability of Ultra-Short-Term", "Threshold-Based Artefact Correction"]
PAPER_GROUP_3 = ["HRV_task_force", "An Overview of Heart Rate Variability Metrics and Norms", "Laborde Mosley Thayer 2017", "Reliability of Ultra-Short-Term", "Threshold-Based Artefact Correction"]
PAPER_GROUP_4 = ["Imburgio et al 2024", "Beggiato et al 2018", "Mahanama et al 2021", "Wang et al 2025", "Wang et al 2024"]
PAPER_GROUP_5 = ["Wang et al 2025", "Wang et al 2024", "Evans et al 2012", "Tomasi et al 2016", "Rupi and Krizek 2019", "Trefzger et al 2018", "Vansteenkiste et al 2014"]
PAPER_GROUP_6 = ["Wang et al 2025", "Wang et al 2024", "Evans et al 2012", "Tomasi et al 2016", "Fleming et al 2024"]

USED = {
    "Heart rate": [("No transformation", ["Zhang et al 2020", "Beggiato et al 2018", "Outdoor temperature heart rate", "Personal_thermal_comfort_wearable_sensors_2018", "Thermal_comfort_physiological_sensing_DTU_2025", "Tervonen"])],
    "Skin temperature": [("No transformation", ["Zhang et al 2020", "Personal_thermal_comfort_wearable_sensors_2018", "Thermal_comfort_physiological_sensing_DTU_2025", "Kyriakou", "novel methodology for human thermal comfort"])],
    "Skin conductance level": [("No transformation", PAPER_GROUP_1)],
    "Skin conductance, tonic": [("No transformation", PAPER_GROUP_1)],
    "Skin conductance, phasic": [("No transformation", ["Fowles", "EDA_recommendations_BIOPAC", "Vocks et al 2021", "EDA_transform_BIOPAC_EDA_SCR", "Figner_Murphy_2010"])],
    "SDNN": [("Natural log", PAPER_GROUP_2)],
    "RMSSD": [("Natural log", PAPER_GROUP_2)],
    "LF power": [("Natural log", PAPER_GROUP_2)],
    "HF power": [("Natural log", PAPER_GROUP_2)],
    "VLF power": [("Natural log", PAPER_GROUP_2)],
    "Total power": [("Natural log", PAPER_GROUP_2)],
    "LF/HF ratio": [("Natural log", PAPER_GROUP_2)],
    "LF normalized": [("No transformation", PAPER_GROUP_3)],
    "HF normalized": [("No transformation", PAPER_GROUP_3)],
    "Pupil diameter": [("No transformation", PAPER_GROUP_4)],
    "Pupil change rate": [("No transformation", PAPER_GROUP_4)],
    "Eyelid aperture": [("No transformation", ["Mahanama et al 2021", "Beggiato et al 2018", "Wang et al 2025", "Wang et al 2024", "Evans et al 2012"])],
    "Blink duration": [("No transformation", PAPER_GROUP_5)],
    "Fixation duration": [("No transformation", PAPER_GROUP_5)],
    "Saccade duration": [("No transformation", PAPER_GROUP_5)],
    "Fixation rate": [("No transformation", PAPER_GROUP_5)],
    "Saccade amplitude": [("No transformation", PAPER_GROUP_5)],
    "Saccade peak velocity": [("No transformation", PAPER_GROUP_5)],
    "Gaze velocity": [("No transformation", PAPER_GROUP_5)],
    "Gaze dispersion": [("No transformation", PAPER_GROUP_5)],
    "Distance from center": [("No transformation", PAPER_GROUP_6)],
    "Gaze centrality": [("No transformation", PAPER_GROUP_6)],
}

COL = {
    "Heart rate": "empatica_heart_rate",
    "Skin temperature": "empatica_skin_temperature",
    "Skin conductance level": "empatica__eda_scl_usiemens",
    "Skin conductance, tonic": "empatica_eda_tonic",
    "Skin conductance, phasic": "empatica_eda_phasic",
    "SDNN": "empatica_hrv_td_sdnn",
    "RMSSD": "empatica_hrv_td_rmssd",
    "LF power": "empatica_hrv_fd_lf",
    "HF power": "empatica_hrv_fd_hf",
    "VLF power": "empatica_hrv_fd_vlf",
    "Total power": "empatica_hrv_fd_total_power",
    "LF/HF ratio": "empatica_hrv_fd_lf_hf_ratio",
    "LF normalized": "empatica_hrv_fd_lfnu",
    "HF normalized": "empatica_hrv_fd_hfnu",
    "Pupil diameter": "eyetracker_pupil_diameter_avg",
    "Pupil change rate": "eyetracker_pupil_change_rate",
    "Eyelid aperture": "eyetracker_eyelid_aperture_avg",
    "Blink duration": "eyetracker_blink_duration_s",
    "Fixation duration": "eyetracker_fixation_duration_s",
    "Saccade duration": "eyetracker_saccade_duration_s",
    "Fixation rate": "eyetracker_fixation_rate",
    "Saccade amplitude": "eyetracker_saccade_amplitude",
    "Saccade peak velocity": "eyetracker_saccade_peak_velocity",
    "Gaze velocity": "eyetracker_gaze_velocity",
    "Gaze dispersion": "eyetracker_gaze_dispersion",
    "Distance from center": "eyetracker_distance_from_center",
    "Gaze centrality": "eyetracker_gaze_centrality",
}

UNIT = {
    "empatica_heart_rate": "bpm", "empatica_skin_temperature": "°C", "empatica__eda_scl_usiemens": "microsiemens",
    "empatica_eda_tonic": "microsiemens", "empatica_eda_phasic": "microsiemens", "empatica_hrv_td_sdnn": "ms",
    "empatica_hrv_td_rmssd": "ms", "empatica_hrv_fd_lf": "ms2", "empatica_hrv_fd_hf": "ms2", "empatica_hrv_fd_vlf": "ms2",
    "empatica_hrv_fd_total_power": "ms2", "empatica_hrv_fd_lf_hf_ratio": "ratio", "empatica_hrv_fd_lfnu": "%",
    "empatica_hrv_fd_hfnu": "%", "eyetracker_pupil_diameter_avg": "mm", "eyetracker_pupil_change_rate": "mm/s",
    "eyetracker_eyelid_aperture_avg": "mm", "eyetracker_blink_duration_s": "s", "eyetracker_fixation_duration_s": "s",
    "eyetracker_saccade_duration_s": "s", "eyetracker_fixation_rate": "per second", "eyetracker_saccade_amplitude": "degrees",
    "eyetracker_saccade_peak_velocity": "degrees/s", "eyetracker_gaze_velocity": "degrees/s",
    "eyetracker_gaze_dispersion": "degrees", "eyetracker_distance_from_center": "pixels", "eyetracker_gaze_centrality": "proportion",
}
assert set(USED) == set(COL), set(USED) ^ set(COL)
assert set(COL.values()) == set(UNIT), set(COL.values()) ^ set(UNIT)
assert {t for opts in USED.values() for t, _ in opts} <= set(STEP5_TEXT)

# Bilateral means used by H4, in addition to every source column.
AVERAGES = {
    **{f"atmo_pm{size}_avg": [f"atmo_lys_atmotube_{side}__atmotube_pm{size}" for side in ["left", "right"]]
       for size in ["1", "2.5", "10"]},
    **{f"lys_{measure}_avg": [f"atmo_lys_LYS{sensor}__lys_{measure}" for sensor in [1, 2]]
       for measure in ["lux", "medi", "kelvin"]},
}
SOUND_PROPORTION_PREFIXES = ("freq_prop_", "octave_prop_")
QC_FIELDS = {"pct_complete", "gvi_match_dist_m", "SVF_Frames_In_Window", "segment", "end_time_s", "phase_start_offset_s"}
METADATA_FIELDS = {"ParticipantID", "PhaseID", "Datetime", "Date"}



REFERENCE_EVIDENCE = {
    "Vocks et al 2021": "File attribution is incorrect: Knejzlikova et al. (2021), doi:10.3389/fpsyg.2021.673597. SCR used a log transform and SCL a square root; not evidence for blanket native-scale EDA.",
    "Figner_Murphy_2010": "Local full text discusses log or square-root treatment of skewed SCR and within-participant standardization; a conditional method discussion, not proof that no transformation is best.",
    "EDA_recommendations_BIOPAC": "Local 2012 recommendations discuss logarithmic or square-root SCR amplitudes; recording in microsiemens is not the same as analysing untransformed values.",
    "Laborde Mosley Thayer 2017": "Normality of HRV Parameters discusses natural-log RMSSD and absolute powers for non-normal HRV; this is general rationale, not variable-by-variable validation of our full model.",
}
print("config ok: %d physio parameters mapped" % len(USED))


config ok: 27 physio parameters mapped


In [2]:
# 2. Load
raw = pd.read_csv(DATA, low_memory=False)
missing_cols = [c for c in COL.values() if c not in raw.columns]
assert not missing_cols, missing_cols

paper_files = tuple(p for p in PAPERS.rglob("*") if p.suffix.lower() == ".pdf")
paper_paths = {}
for key in dict.fromkeys(key for options in USED.values() for _, keys in options for key in keys):
    hits = [p for p in paper_files if key in p.name]
    if len(hits) != 1:
        raise FileNotFoundError(f"{key} -> {[str(p) for p in hits]}")
    paper_paths[key] = hits[0]
core_values = raw.loc[raw.PhaseID.isin(CORE_PHASES), list(COL.values())].apply(pd.to_numeric, errors="coerce")
nonpositive_counts = core_values.le(0).sum()

normalization_columns = sorted(c for c in raw.columns if in_normalization_scope(c))
core = raw.loc[raw.PhaseID.isin(CORE_PHASES)].copy()
print("loaded %d rows (%d core phases), %d literature PDFs resolved, %d normalization columns" % (len(raw), len(core), len(paper_paths), len(normalization_columns)))


loaded 5077 rows (2156 core phases), 28 literature PDFs resolved, 41 normalization columns


In [3]:
# 3. Define
def step5_text(col_name, transformation):
    return "Not applicable" if col_name in NOT_IN_STEP5 else STEP5_TEXT[transformation]

def check_log_domain(col_name, transformation):
    return int(nonpositive_counts[col_name]) if transformation == "Natural log" else 0


def make_normalization_tables():
    catalog = []
    number_of = {}
    rows = []
    physio_rows = []
    for parameter, options in USED.items():
        transformation, keys = options[0]
        marks, titles = [], []
        for key in keys:
            path = paper_paths[key]
            if path.name not in number_of:
                number_of[path.name] = len(catalog) + 1
                catalog.append({"Number": f"[{number_of[path.name]}]", "Paper title": path.name, "File path": str(path), "Evidence checked": REFERENCE_EVIDENCE.get(key, "Existing mapping; not newly full-text verified")})
            marks.append(f"[{number_of[path.name]}]")
            titles.append(path.name)
        col_name = COL[parameter]
        rows.append({
            "col_name": col_name,
            "Parameter": parameter,
            "Transformation": transformation,
            "Number of papers": len(marks),
            "Papers": ", ".join(marks),
        })
        physio_rows.append({
            "Column": col_name,
            "Parameter": parameter,
            "Unit": UNIT[col_name],
            "Transformation in the papers": ("Mixed / conditional: SCR log or square root; SCL square root in the checked 2021 paper" if "conductance" in parameter.lower() else "Natural-log HRV rationale; variable-specific mapping not fully verified" if transformation == "Natural log" else "Existing mapped references; native-scale optimum not established"),
            "Transformation in Step 5": step5_text(col_name, transformation),
            "Rows excluded (x <= 0)": check_log_domain(col_name, transformation),
            "Paper titles": " | ".join(titles),
        })
    
    return pd.DataFrame(rows), pd.DataFrame(catalog), pd.DataFrame(physio_rows)

def transformation_policy(column, series):
    if column in COL.values():
        parameter = next(p for p, c in COL.items() if c == column)
        transformation = USED[parameter][0][0]
        return "Physiological outcome", STEP5_TEXT[transformation], "Existing literature mapping; transform before model-specific standardization"
    if column in METADATA_FIELDS or column.endswith("_Date") or column.startswith("ucm_GPS_") or column == "ucm_MAG_hdg":
        return "Metadata", "Not applicable", "Identifier, timestamp or positioning metadata; not a continuous exposure transformation"
    if column in QC_FIELDS or column.startswith("empatica_hrv_fd_5min_") or column.startswith("SVF_") and column != "SVF":
        return "QC / processing metadata", "Not applicable", "Availability, quality or processing descriptor; retained for checking"
    if column == "unidentify_prop":
        return "Classification diagnostic", "Not applicable to exposure model", "Unidentified sound share; excluded before sound PCA"
    if column == "category" or column.endswith(("_SR", "_lux_cat", "__lux_cat")) or column == "ucm_IO_flag":
        return "Categorical / ordinal code", "No continuous transformation", "Use declared category coding; numerical storage does not imply a continuous measurement"
    if column.startswith("questionnaire_"):
        score = any(x in column for x in ["_mean_", "_comfort_", "_pleasantness_", "_familiarity_"])
        return ("Questionnaire score" if score else "Questionnaire / participant metadata", "Native score or categorical coding; no automatic log", "Repeated questionnaire values do not become independent ten-second observations; model depends on item/score type")
    if column.endswith("_%") or column in {"SVF", "gvi_50m"} or column.startswith(("gvi_", "tree_phase_", "grass_phase_", "plant_phase_")):
        return "SVF / greenery exposure", "Native scale before model-specific centering/scaling", "Bounded exposure; no automatic log/logit. Boundary values and model role must be checked"
    if column == "ucm_SND_dBA" or column.startswith(SOUND_PROPORTION_PREFIXES) or column.endswith(("_score", "_prop")) or column in {"dominant_frequency_hz", "spectral_centroid_hz", "spectral_rolloff85_hz"}:
        return "Sound exposure", "Native scale; within-bout centering and unit-SD scaling for sound PCA", "dBA is already logarithmic; do not log again. Proportions/scores are not probabilities of equivalent meaning; current PCA is correlation-based"
    if pd.api.types.is_numeric_dtype(series):
        return "Numeric measurement / derived field", "Native scale before model-specific centering/scaling", "Native scale; within-bout centering and pooled-SD scaling in H4 when entered"
    return "Text / categorical field", "No continuous transformation", "Preserve labels; no z-score or numeric log of text"

def make_column_inventory(physio_counts):
    existing = physio_counts.set_index("col_name")
    rows = []
    for column in normalization_columns:
        values = core[column]
        role, treatment, basis = transformation_policy(column, values)
        numeric = pd.to_numeric(values, errors="coerce") if pd.api.types.is_numeric_dtype(values) else pd.Series(dtype=float)
        valid = numeric.dropna()
        ref = existing.loc[column].to_dict() if column in existing.index else {}
        rows.append({"col_name": column, "Parameter": ref.get("Parameter", column),
            "Role": role, "Transformation": treatment, "Basis": basis,
            "Scaling": "Model-specific; metadata/QC/categorical fields are not automatically standardized",
            "Number of papers": ref.get("Number of papers", 0), "Papers": ref.get("Papers", ""),
            "Origin": "Bilateral mean" if column in AVERAGES else "Merged source",
            "Core rows": len(core), "Missing": int(values.isna().sum()), "Unique": values.nunique(),
            "Nonpositive": int((valid <= 0).sum()) if len(valid) else None,
            "Minimum": valid.min(), "Maximum": valid.max(),
            "Skewness": valid.skew() if len(valid) >= 3 and valid.nunique() > 1 else None,
            "Nonfinite": int((~np.isfinite(numeric.dropna())).sum()) if len(numeric) else 0})
    table = pd.DataFrame(rows)
    assert not table.col_name.duplicated().any()
    assert set(table.col_name) == set(normalization_columns)
    return table

print("functions defined (normalization tables + column inventory)")


functions defined (normalization tables + column inventory)


In [4]:
def write_csv(path, frame):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=OUT_DIR, suffix='.tmp', delete=False) as handle:
        tmp = Path(handle.name)
        frame.to_csv(handle, index=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(tmp, path)
    print(path.resolve())

OUT_ALL = OUT_DIR / 'Normalization_H1toH4_all_parameters.csv'
OUT_DIAG = OUT_DIR / 'Normalization_H1toH4_diagnostics.csv'

physio_counts, papers_table, physio = make_normalization_tables()
count = make_column_inventory(physio_counts)
phys = physio.rename(columns={
    'Transformation in Step 5': 'transformation_in_models',
    'Paper titles': 'paper_citations',
})[['Column', 'transformation_in_models', 'paper_citations']]
all_params = count.rename(columns={
    'col_name': 'column_name',
    'Parameter': 'parameter_label',
    'Role': 'role',
    'Transformation': 'transformation_in_models_default',
    'Number of papers': 'number_of_papers',
    'Papers': 'paper_keys',
}).merge(phys, left_on='column_name', right_on='Column', how='left')
all_params['transformation_in_models'] = all_params['transformation_in_models'].fillna(all_params['transformation_in_models_default'])
all_params['paper_citations'] = all_params['paper_citations'].fillna('')
all_params = all_params[all_params['column_name'].map(in_normalization_scope)]
all_params = all_params[[
    'column_name', 'parameter_label', 'role', 'transformation_in_models',
    'number_of_papers', 'paper_keys', 'paper_citations',
    'Missing', 'Nonpositive', 'Skewness',
]].sort_values(['role', 'column_name'])
diag = count[count['col_name'].map(in_normalization_scope)][['col_name', 'Role', 'Missing', 'Nonpositive', 'Minimum', 'Maximum', 'Skewness', 'Nonfinite']].rename(columns={'col_name': 'column_name'})
write_csv(OUT_ALL, all_params)
write_csv(OUT_DIAG, diag)
print('all_parameters=%d with_papers=%d' % (len(all_params), int((all_params['number_of_papers'].fillna(0) > 0).sum())))


C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_H1toH4_all_parameters.csv
C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_H1toH4_diagnostics.csv
all_parameters=41 with_papers=27
